# C7-cnn-transfer — Practice p10

**Type:** constrained coding · **Difficulty:** core · **Concepts:** layer-freezing, nn-module, requires-grad, parameter-counting, cnn-training

**Time budget:** 75 minutes

Use only CPU tensors. The supplied tiny model and synthetic batch use seed
`20260804`; do not change their definitions, draw order, batching, or
hyperparameters.

## Part I — selective freezing and two-currency audit (8 points)

Freeze **exactly** every parameter whose qualified name starts with `conv1`,
`bn1`, `layer1`, or `layer2`; every other parameter remains trainable. Use an
explicit filter over `model.named_parameters()` and assign:

- `n_frozen_tensors`, `n_frozen_scalars`, `n_trainable_scalars`;
- `split_ok`, true exactly when frozen plus trainable scalars equals the model
  total; and
- `trainable_tops`, the sorted top-level prefixes owning trainable parameters.

## Part II — train only what the audit permits (12 points)

After Part I, snapshot every named parameter and construct `optimizer` from
**only** trainable parameter objects. Train for exactly **18 full-batch steps**
in the fixed order
`zero_grad(set_to_none=True) → forward → loss → backward → step`, using the
supplied `CrossEntropyLoss`, data, and Adam (`lr=0.05`) hyperparameters. Assign:

- `loss_history` (Python floats measured before each update);
- `optimizer_owns_exactly_trainable` using parameter-object identity;
- `gradient_names`, the sorted parameter names whose gradients are not `None`
  after the final backward;
- `moved_trainable_names`, the sorted trainable names that moved;
- `frozen_bitwise_unchanged`; and
- `training_certificate`, true only when loss is finite, final loss is at most
  `0.80 * loss_history[0]`, optimizer ownership is exact, gradients occur
  exactly on trainable names, at least one trainable parameter moves, and every
  frozen parameter is bitwise unchanged (`torch.equal`).

**Banned — zero points for Part I:** `model.requires_grad_(...)`,
`torchsummary`, `torchinfo`, `p.nelement()`, or replacing the explicit
name-filter/count audit with a summary API.

**Banned — zero points for Part II:** pretrained weights, downloads/network,
changing the supplied model/data/seed/order/steps/hyperparameters, including a
frozen parameter in the optimizer, manually editing `.grad` or parameter
`.data`, or reporting only loss without the ownership/gradient/movement
certificate.

In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)
SEED = 20260804
torch.manual_seed(SEED)

class TinyTransferCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 3, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(3)
        self.layer1 = nn.Sequential(nn.Conv2d(3, 4, 3, padding=1), nn.ReLU())
        self.layer2 = nn.Sequential(nn.Conv2d(4, 5, 3, padding=1), nn.ReLU())
        self.layer3 = nn.Sequential(nn.Conv2d(5, 6, 3, padding=1), nn.ReLU())
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(6, 3)

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.fc(torch.flatten(self.pool(x), 1))

generator = torch.Generator(device="cpu").manual_seed(SEED)
X = 0.05 * torch.randn(18, 1, 6, 6, generator=generator)
y = torch.arange(18, dtype=torch.long) % 3
X[y == 0, :, :, 1:3] += 1.0
X[y == 1, :, 3:5, :] += 1.0
diagonal = torch.arange(6)
class_two = X[y == 2].clone()
class_two[:, :, diagonal, diagonal] += 1.0
X[y == 2] = class_two
X[y == 0] -= 0.8
X[y == 2] += 0.8
model = TinyTransferCNN()
criterion = nn.CrossEntropyLoss()


In [ ]:
# Part I: explicit selective-freezing loop
# YOUR CODE HERE

n_frozen_tensors = ...
n_frozen_scalars = ...
n_trainable_scalars = ...
split_ok = ...
trainable_tops = ...

# Part II: snapshot, construct optimizer, and train exactly 18 full-batch steps
parameter_before = ...
optimizer = ...
loss_history = ...
optimizer_owns_exactly_trainable = ...
gradient_names = ...
moved_trainable_names = ...
frozen_bitwise_unchanged = ...
training_certificate = ...
